In [ ]:
# 1 — GPU check
import os,platform,shutil,subprocess,sys
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(),'CUDA GPU required'
print(platform.platform(),torch.cuda.get_device_name(0),torch.version.cuda,sys.version)

In [ ]:
# 2 — Pinned dependencies and torchao cleanup
import importlib.util,subprocess,sys
if importlib.util.find_spec('torchao') is not None: subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.20.0','accelerate','safetensors','psutil'],check=True)
os.environ['WANDB_DISABLED']='true';os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
# 3 — Mount persistent storage before training
from google.colab import drive
drive.mount('/content/drive')
PERSIST_ROOT='/content/drive/MyDrive/PlannerAgent/GCC4K'
os.environ['PLANNERAGENT_GCC4K_PERSIST_ROOT']=PERSIST_ROOT
os.makedirs(PERSIST_ROOT,exist_ok=True)

In [ ]:
# 4 — Upload recovery input bundle
from google.colab import files
uploaded=files.upload();archives=[name for name in uploaded if name.endswith('.zip')]
assert len(archives)==1,'Upload exactly one GCC4K RECOVERY INPUT ZIP'
shutil.rmtree('/content/gcc4k',ignore_errors=True);shutil.unpack_archive(archives[0],'/content/gcc4k')

In [ ]:
# 5 — Verify input hashes and paths
import hashlib,pathlib
root=pathlib.Path('/content/gcc4k')
for line in (root/'SHA256SUMS.txt').read_text().splitlines():
    expected,relative=line.split('  ',1);assert '\\' not in relative;assert hashlib.sha256((root/relative).read_bytes()).hexdigest()==expected,relative
print('INPUT INTEGRITY PASS')

In [ ]:
# 6 — Compatibility/API preflight occurs before model download
!cd /content/gcc4k/scripts && python train_targeted_student_v02.py --help
import transformers,peft,inspect
from transformers import Trainer,TrainingArguments
assert transformers.__version__=='4.51.3';assert 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters;assert 'eval_dataset' in inspect.signature(Trainer.__init__).parameters

In [ ]:
# 7 — Show persistent recovery state
candidate=pathlib.Path(PERSIST_ROOT)/'PA-INTERPRETATION-STUDENT-v0.2'
for name in ('state','checkpoints','final-adapter','evaluation','result'):
    path=candidate/name;print(name,'exists' if path.exists() else 'missing',list(path.iterdir())[-3:] if path.exists() else '')

In [ ]:
# 8 — TRAIN only; resumes checkpoint or skips verified final adapter
!cd /content/gcc4k/scripts && python train_targeted_student_v02.py --phase train

In [ ]:
# 9 — Confirm persistent final adapter and hashes before evaluation
final_adapter=candidate/'final-adapter'
assert (final_adapter/'COMPLETE').is_file();manifest=__import__('json').loads((final_adapter/'candidate.manifest.json').read_text())
for relative,expected in manifest['artifact_hashes'].items():assert hashlib.sha256((final_adapter/relative).read_bytes()).hexdigest()==expected
print('PERSISTENT FINAL ADAPTER PASS',manifest['adapter_digest'])

In [ ]:
# 10 — EVALUATE only; skips every verified completed dataset
!cd /content/gcc4k/scripts && python train_targeted_student_v02.py --phase evaluate

In [ ]:
# 11 — Verify Drive-first final result ZIP
result=candidate/'result'/'PA-INTERPRETATION-STUDENT-v0.2-GCC4K.zip';result_manifest=__import__('json').loads((candidate/'result'/'final-result.manifest.json').read_text())
assert (candidate/'result'/'FINAL_RESULT_COMPLETE').is_file();assert hashlib.sha256(result.read_bytes()).hexdigest()==result_manifest['zip_sha256'];print(result,result.stat().st_size,result_manifest['zip_sha256'])

In [ ]:
# 12 — Browser download is optional; Drive remains authoritative
files.download(str(result))